# Datamine SELTRI Process Example

This notebook demonstrates how to configure and run the **`seltri`** process wrapper in `dmstudio`.

## Process Description

## SELTRI

Copies records of a file which have X, Y and Z co-ordinates lying above/ below a wireframe surface or inside/outside a wireframe volume.

* **Note** (Surfaces and wireframes should not both be present in the triangles file unless individual models are specified for selection by setting a **ZONE** parameter. In-place operations are not permitted.):

### @SELECT Options

## * **@SELECT = 1 OR 2**

There may be points that are neither above or below if they are outside the range of triangles.

## * **@SELECT = 3**

A point may lie inside more than one wireframe model. If this is the case it will appear more than once in the output file.

## * **@SELECT = 4**

An outside point is outside all wireframe models. Attribute fields will not be copied in this case.

### Input Files:

* **in** (Undefined):
  Input point file for selection. Must have explicit numeric fields X , Y and Z.
  Processing is quicker if this file is sorted on X.
  Required=Yes

* **wiretr** (Wireframe Triangle):
  Input wireframe triangle file
  Required=Yes

* **wirept** (Wireframe Points):
  Input wireframe point file
  Required=Yes

### Output Files:

* **out** (Undefined):
  Output file of selected records. File may contain additional fields, including the
  **ZONE** field.
  Required=Yes

### Fields:

* **x** (Numeric : IN):
  Field in **IN** file defining the X co-ordinate.
  Default=Undefined
  Required=Yes

* **y** (Numeric : IN):
  Field in **IN** file defining the Y co-ordinate.
  Default=Undefined
  Required=Yes

* **z** (Numeric : IN):
  Field in **IN** file defining the Z co-ordinate.
  Default=Undefined
  Required=Yes

* **zone** (Any : WIRETR):
  Field in **WIRETR** file used to identify individual surfaces and solid models. If
  there is more than one surface or solid model and **ZONE** is not specified, a point may
  be selected more than once, one for each model. If this field is specified and **ZONE**
  is not, the **ZONE** field is copied to the output file.
  Default=Undefined
  Required=No

* **attribs** (Undefined : Undefined):
  Field from the **WIRETR** file to be placed into the output file for all records which are selected. Up to 4 words may be entered, which may be 4 numeric fields or a mixture of alphanumeric and numeric fields totalling 4 words.
  Default=Undefined
  Required=No

### Parameters:

* **zone**:
  Used to select one of a number of surface or wireframe models. This parameter must be
  specified if **WIRETR** contains a mix of surface and solid models. The SELECT parameter
  will determine whether a model is treated as a surface or wireframe.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **select**:

* **Options** (1: select points above a surface; 2: select points below a surface; 3: select):
  points inside a solid; 4: select points outside a solid.
  Range=1,4
  Values=1,2,3,4
  Default=3
  Required=Yes

* **toleranc**:
  Tolerance for selection criteria. A positive tolerance will enhance point selection
  (0.001).
  Range=Undefined
  Values=Undefined
  Default=0.001
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('seltri')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute seltri
print("Running seltri...")
dm_cmd.seltri(
    in_i='t_assays',  # required
    wiretr_i='t_SurfaceTriangles',  # required
    wirept_i='t_SurfaceTriangles',  # required
    out_o='t_seltri_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    z_f='Z',  # required
    select_p='required_val',  # required
    # zone_f='optional',  # optional
    # attribs_f=['optional'],  # optional
    # zone_p='optional',  # optional
    # toleranc_p=0.001,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("seltri execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_seltri_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")